In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Central Difference Convolutional Layer (CDCLayer)
class CDCLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(CDCLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.central_diff = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        conv_out = self.conv(x)
        diff_out = self.central_diff(x)
        x = conv_out - diff_out
        x = self.bn(x)
        return self.dropout(F.relu(x))

# Multi-Attentional Feature Map (MAFM)
class MAFM(nn.Module):
    def __init__(self, in_channels):
        super(MAFM, self).__init__()
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // 4, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels // 4, in_channels, kernel_size=1),
            nn.Sigmoid()
        )
        self.spatial_att = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Channel Attention
        channel_weights = self.channel_att(x)
        x = x * channel_weights
        
        # Spatial Attention
        spatial_weights = self.spatial_att(torch.cat([
            torch.mean(x, dim=1, keepdim=True),
            torch.max(x, dim=1, keepdim=True)[0]
        ], dim=1))
        return x * spatial_weights

# Spoof Cue Map Generator
class SpoofCueMapGenerator(nn.Module):
    def __init__(self):
        super(SpoofCueMapGenerator, self).__init__()

    def forward(self, x):
        batch_size, _, height, width = x.shape
        # Generate random spoof cue maps
        cue_map = torch.randn((batch_size, 1, height, width)).to(x.device)
        return cue_map

# Feature Fusion Module
class FeatureFusionModule(nn.Module):
    def __init__(self, in_channels):
        super(FeatureFusionModule, self).__init__()
        self.conv = nn.Conv2d(in_channels + 1, in_channels, kernel_size=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, features, cue_map):
        # Concatenate features and cue map
        fused_features = torch.cat([features, cue_map], dim=1)
        fused_features = self.relu(self.conv(fused_features))
        return self.dropout(fused_features)

# Enhanced CDCN++ Model with All Modules
class CDCNPlusPlus(nn.Module):
    def __init__(self):
        super(CDCNPlusPlus, self).__init__()
        
        # Multiple CDC Layers for Feature Extraction
        self.cdc1 = CDCLayer(3, 32)
        self.cdc2 = CDCLayer(32, 64)
        self.cdc3 = CDCLayer(64, 128)
        self.cdc4 = CDCLayer(128, 128)
        self.cdc5 = CDCLayer(128, 64)
        
        self.mafm = MAFM(64)  # Multi-Attentional Feature Map
        self.spoof_cue_map_gen = SpoofCueMapGenerator()  # Generate spoof cue maps
        self.feature_fusion = FeatureFusionModule(64)  # Fuse features and cue maps
        
        self.pool = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        self.dropout = nn.Dropout(0.4)  # Regularization
        self.fc = nn.Linear(64, 1)  # Binary classification output
        
    def forward(self, x):
        # Pass through multiple CDC Layers
        x = self.cdc1(x)
        x = self.cdc2(x)
        x = self.cdc3(x)
        x = self.cdc4(x)
        x = self.cdc5(x)
        
        # Apply Multi-Attentional Feature Map (MAFM)
        x = self.mafm(x)
        
        # Generate Spoof Cue Map and Perform Feature Fusion
        cue_map = self.spoof_cue_map_gen(x)
        x = self.feature_fusion(x, cue_map)
        
        # Global Average Pooling and Classification
        x = self.pool(x).view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

model = CDCNPlusPlus()
print(model)

CDCNPlusPlus(
  (cdc1): CDCLayer(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (central_diff): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (cdc2): CDCLayer(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (central_diff): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (cdc3): CDCLayer(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (central_diff): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=Tru

In [ ]:

import cv2
import torch
import numpy as np
from torchvision import transforms
from src.cdcn_module import CDCNPlusPlusClassifier 
import torch.nn.functional as F

# Load the model
model_path = "models/fasecure_print_replay.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CDCNPlusPlusClassifier()
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# Face detection model (OpenCV)
#reading from caffee model is no longer present in the latest stable cv2 hence older version
net = cv2.dnn.readNetFromCaffe("resources/deploy.prototxt.txt", "resources/res10_300x300_ssd_iter_140000.caffemodel")


# Webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Define transform for input image
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break

    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()

    face_detected = False
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5:
            face_detected = True
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            input_tensor = preprocess(face).unsqueeze(0).to(device)
            with torch.no_grad():
                output = model(input_tensor)
                prob = torch.sigmoid(output).item()
                label = "Spoof" if prob > 0.5 else "Live"
                color = (0, 0, 255) if label == "Spoof" else (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label}: {prob:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if not face_detected:
        cv2.putText(frame, "No face detected", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("FAS - Real Time", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


cpu
Initializing CDCN++ Model with Central Difference Convolution Layers


c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
